# Book QA pilot: safe aggregate checks
One row per candidate. This notebook reads no private book or benchmark text. Automatic approval is not human acceptance. No temporal comparison is available.

In [ ]:
import json
from pathlib import Path
path = Path('book_sft_api_pilot_20260907.safe.json')
if not path.exists():
    path = Path('docs') / path
s = json.loads(path.read_text())
assert sum(s['status'].values()) == s['completed'] == 200
assert sum(s['kind_counts'].values()) == 200
assert s['review_queue_count'] + s['overlap_quarantined'] == s['status']['auto_pass']
assert s['usage']['prompt_tokens'] + s['usage']['completion_tokens'] == s['usage']['total_tokens']
assert not s['ready_for_training'] and not s['human_review_complete']
print({k: {'count': v, 'fraction': v/s['completed']} for k, v in s['status'].items()})
print('Lexical screen only; no proof of semantic decontamination.')

In [ ]:
r = json.loads((path.parent / 'book_sft_api_review_20260907.safe.json').read_text())
assert sum(sum(g.values()) for g in r['groups'].values()) == r['sample_n'] == 28
assert len(set(r['sample_ids'])) == 28
assert sum(r['groups']['auto_pass'].values()) == 20
assert sum(r['all_structural_reject_quote_status'].values()) == 32
assert r['usage']['prompt_tokens'] + r['usage']['completion_tokens'] == r['usage']['total_tokens']
assert not r['human_review_complete'] and not r['expanded'] and not r['training_started']
print(r['groups'])
print('Re-review disagreement is not an established factual error rate.')

In [ ]:
a = json.loads((path.parent / 'book_sft_api_adjudication_20260907.safe.json').read_text())
assert sum(a['status'].values()) == a['completed'] == 8
assert len(a['per_item']) == a['status']['valid_adjudication']
assert sum(sum(x['categories'].values()) for x in a['per_item']) == sum(a['issue_categories'].values())
assert a['usage']['prompt_tokens'] + a['usage']['completion_tokens'] == a['usage']['total_tokens']
assert not a['automatic_promotion'] and not a['training_started'] and not a['expanded']
print(a['status'], a['issue_categories'])
print('Categories describe model judgements, not established factual error counts.')